# 06장 보안 실습 — 프로세스·소켓 문맥 연결


## Goal

합성 스냅샷을 연결하고 PID·시각·서비스 문맥의 한계를 설명합니다.

[교안과 분석 질문](../../06-system-inspection/06-3-host-process-investigation.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-06-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'processes.psv': 'pid|ppid|user|started_kst|exe|unit\n1|0|root|2026-09-10T08:00:00+09:00|/usr/lib/systemd/systemd|init.scope\n200|1|root|2026-09-10T08:01:00+09:00|/usr/sbin/sshd|ssh.service\n410|200|analyst|2026-09-10T09:02:00+09:00|/usr/bin/bash|session-4.scope\n520|1|collector|2026-09-10T09:05:00+09:00|/opt/collector/bin/report|report-helper.service\n', 'sockets.psv': 'state|local|peer|pid\nLISTEN|0.0.0.0:22|0.0.0.0:*|200\nESTAB|192.0.2.20:22|192.0.2.10:50103|200\nESTAB|192.0.2.20:41000|203.0.113.7:443|520\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: 프로세스 4행, PID 520의 서비스·목적지 연결, 판정은 추가 검토

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 1. 필드와 프로세스 수 확인


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
head -n 1 "$COURSE_DATA/processes.psv"
awk -F '|' 'NR>1 {print $1, $2, $3, $5}' "$COURSE_DATA/processes.psv" > "$COURSE_OUT/process-preview.txt"
test "$(wc -l < "$COURSE_OUT/process-preview.txt")" -eq 4
printf 'process_rows=4\n'


### 2. PID로 연결하되 시각 한계를 유지


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR==FNR {if (FNR>1) exe[$1]=$5; next}
 FNR>1 {print $1 "|" $3 "|" $4 "|" (($4 in exe) ? exe[$4] : "unknown")}' \
 "$COURSE_DATA/processes.psv" "$COURSE_DATA/sockets.psv" > "$COURSE_OUT/linked.psv"
grep -Fx 'ESTAB|203.0.113.7:443|520|/opt/collector/bin/report' "$COURSE_OUT/linked.psv"


### 3. 부모와 서비스 문맥 확인


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 && $1==520 {print "parent=" $2 " user=" $3 " unit=" $6}' \
 "$COURSE_DATA/processes.psv" > "$COURSE_OUT/context.txt"
grep -Fx 'parent=1 user=collector unit=report-helper.service' "$COURSE_OUT/context.txt"
printf 'verdict=needs_service_and_destination_review\n'


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
